In [ ]:
from openai import OpenAI
import pandas as pd
import time

In [ ]:
new_data = pd.read_csv('/GEO_metadata_cleaned.csv')

In [ ]:
# Initialize OpenAI client
client = OpenAI(
  api_key="#api_key#")


selected_rows = new_data[]

# Create an empty list to store results
results = []

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Define a function to generate the prompt and make the API call
def process_row(row):
    prompt = f"""You are an expert in clinical informatics and genomic data analysis. Your task is to analyze the clinical metadata associated with a microarray sample (derived from platforms GPL570, GPL96, or GPL97) and
    determine the most appropriate ICD-10 code for the primary diagnosis. In addition, you must provide a certainty score indicating your confidence in the assigned code on a scale from 0 to 100 (where 100 means absolute certainty).
    The input data will be in format of Experiment (where several samples are part of one experiment) information and Sample Description. 
        Please follow these guidelines:

            Output Format:
            Return your output strictly in the following format (no extra text or explanation):
            [ICD-10 code], [certainty score]
            Example: E11.9, 85

            Healthy Cases:
            If the clinical metadata does not indicate any disease or abnormality, output the ICD-10 code for a healthy state, which is Z00.00, along with your certainty score.
            Example: Z00.00, 100

            Ambiguity Handling:
            If the clinical metadata is ambiguous or insufficient to assign a specific ICD-10 code with reasonable certainty, output Unknown with a certainty score of 0.
            Example: Unknown, 0

            Validity:
            Ensure that the ICD-10 code you provide is valid according to the official ICD-10 coding system, and base your decision solely on the provided clinical metadata.

        Input: {row.Data}
        Output Example: E11.9, 85
    """

    try:
        completion = client.chat.completions.create(
            model="o3-mini",
            messages=[{"role": "user", "content": prompt}],
            reasoning_effort="high"
        )
        icd_code = completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error processing LOCAL_ID {row.LOCAL_ID}: {e}")
        icd_code = "Unknown, 0"

    print(f"LOCAL_ID: {row.LOCAL_ID} - ICD-10 Code: {icd_code}")
    return (row.LOCAL_ID, icd_code)


# Example: select subset of rows first
# selected_rows = new_data.iloc[0:500]  # safer for DataFrame slicing

# For concurrent processing
with ThreadPoolExecutor(max_workers=10) as executor:
    future_to_row = {executor.submit(process_row, row): row for row in selected_rows.itertuples()}
    for future in as_completed(future_to_row):
        results.append(future.result())

# Create DataFrame and save to CSV
icd_df_o3_low = pd.DataFrame(results, columns=['LOCAL_ID', 'ICD_code'])
icd_df_o3_low.to_csv('icd_codes_output_o3_low.csv', index=False)
print("✅ ICD-10 coding completed and saved to icd_codes_output_o3_low.csv")


In [ ]:
icd_df_o3_low = pd.DataFrame(results, columns=['LOCAL_ID', 'ICD_code'])
icd_df_o3_low.to_csv('icd_codes_output_o3_low.csv', index=False)
print("✅ ICD-10 coding completed and saved to icd_codes_output_o3_low.csv")